# Incident triage with Daybreak Blue

| | |
| --- | --- |
| **Model** | `openai.gpt-daybreak-blue-5.6-sol` |
| **Inference provider** | Amazon Bedrock |
| **Problem** | Turn synthetic cloud identity events into an evidence-linked incident brief |
| **Region** | `us-east-2` (Ohio) by default; set `AWS_REGION` to use another approved Region |
| **Discovery endpoint** | `https://bedrock-mantle.us-east-2.api.aws/v1` |
| **Inference endpoint** | `https://bedrock-mantle.us-east-2.api.aws/openai/v1` |
| **Difficulty** | Beginner |

## What you will take away

By the end, you will have verified Daybreak Blue access through Amazon Bedrock, turned five synthetic identity events into a structured incident brief, and applied a practical review checklist before any analyst decision or containment action.

Daybreak Blue uses GPT-5.6 Sol as the broad defensive tier. It is the usual starting point for vulnerability discovery and triage, secure code review, detection engineering, incident response, controlled malware analysis, and patch validation. This notebook uses Blue because the job is to assess supplied evidence and recommend next steps, not reproduce an exploit.

Daybreak Red uses the specialist GPT-5.6 Cyber model. It is separately approved for more advanced work such as controlled vulnerability reproduction, proof-of-concept or exploit validation, penetration testing, red teaming, and complex system analysis. Red is not the default choice for routine defensive work.

This is a synthetic proof of concept: five invented cloud identity events go in, and an evidence-linked incident brief comes back. The model has no tools and cannot take containment action. Do not replace the fixture with customer logs, credentials, personal data, or production identifiers.

Daybreak access is gated. See [Trusted Access for Cyber](https://learn.chatgpt.com/docs/cyber-safety), the [Blue model card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-openai-gpt-daybreak-blue-56-sol.html), and the recipe [README](../README.md) before running the notebook.

## Set up the client and load the sample

From `cookbooks/`, run `uv sync --group cybersecurity` and use that environment as the kernel. Set `AWS_REGION=us-east-2`; if you use a named profile, also set `AWS_PROFILE`, then run `aws sts get-caller-identity` and confirm the account and role before opening the notebook.

The AWS token generator derives refreshable short-term Bedrock credentials from the standard AWS credential chain. The generic client below uses `/v1` only for model discovery; `BedrockOpenAI` uses `/openai/v1` for inference. The cell also finds the fixture whether the notebook is opened from the recipe folder, `cookbooks/`, or the repository root.

In [ ]:
import json
import os
from pathlib import Path

from aws_bedrock_token_generator import provide_token
from openai import BedrockOpenAI, OpenAI

REGION = (
    os.environ.get("AWS_REGION")
    or os.environ.get("AWS_DEFAULT_REGION")
    or "us-east-2"
)
HOST = f"https://bedrock-mantle.{REGION}.api.aws"
CATALOG_ENDPOINT = f"{HOST}/v1"
INFERENCE_ENDPOINT = f"{HOST}/openai/v1"
MODEL_ID = os.environ.get("MODEL_ID", "openai.gpt-daybreak-blue-5.6-sol")
PROBE_MAX_OUTPUT_TOKENS = 128
MAX_OUTPUT_TOKENS = 1200

def bedrock_token() -> str:
    return provide_token(region=REGION)


catalog = OpenAI(
    api_key=bedrock_token(),
    base_url=CATALOG_ENDPOINT,
    max_retries=3,
)
client = BedrockOpenAI(
    aws_region=REGION,
    bedrock_token_provider=bedrock_token,
    max_retries=3,
)

relative_data = Path(
    "06-cybersecurity/01-daybreak-blue-incident-triage/data/identity_events.jsonl"
)
candidates = (
    Path.cwd() / "data" / "identity_events.jsonl",
    Path.cwd().parent / "data" / "identity_events.jsonl",
    Path.cwd() / relative_data,
    Path.cwd() / "cookbooks" / relative_data,
)
data_file = next(path for path in candidates if path.exists())
events = [json.loads(line) for line in data_file.read_text().splitlines() if line]

print(f"{len(events)} fabricated events · {MODEL_ID} in {REGION}")
print("Discovery endpoint:", CATALOG_ENDPOINT)
print("Inference endpoint:", INFERENCE_ENDPOINT)
print("Event IDs:", [event["event_id"] for event in events])

## Discover the model

The Mantle Models API lists models available to this AWS environment. This catches a wrong Region or missing model before any workload request. Discovery is necessary but not sufficient: the next cell still performs a real inference check.

In [ ]:
model_ids = {model.id for model in catalog.models.list().data}
if MODEL_ID not in model_ids:
    raise RuntimeError(
        f"{MODEL_ID} was not returned by {CATALOG_ENDPOINT}/models. "
        "Confirm model approval, AWS identity, and Region."
    )

print(f"Model discovery passed: {MODEL_ID}")

## Check model access

This is a real, small inference request using the same model, Region, endpoint, and credentials as the incident-triage call. Run it before continuing. If approval, credentials, IAM permissions, model access, or the Region is wrong, the cell should fail here with the SDK error. The request is billable and uses `store=False`.

In [ ]:
probe = client.responses.create(
    model=MODEL_ID,
    input="Reply with READY to confirm that basic inference is working.",
    max_output_tokens=PROBE_MAX_OUTPUT_TOKENS,
    store=False,
)

probe_text = probe.output_text.strip()
if probe.status != "completed" or not probe_text:
    raise RuntimeError(
        f"Model check did not complete successfully: id={probe.id}, "
        f"status={probe.status!r}"
    )

print(f"Model check passed: {probe_text}")

## Ask for an incident brief

The output shape is fixed so an analyst can quickly check the evidence, confidence, alternatives, and next steps. The request sets `store=False`; confirm retention separately because that setting is not the same as Zero Data Retention.

In [ ]:
instructions = """You are assisting an authorized defensive security team.
Analyze only the synthetic events supplied in this request. Do not infer real identities
or claim access to systems or telemetry that is not shown. Keep evidence, inference, and
recommended action distinct. Prefer reversible containment and human approval."""

task = f"""Create a concise incident-triage brief from these synthetic events:

{json.dumps(events, indent=2)}

Use these headings:
1. Assessment and confidence
2. Evidence timeline (cite event IDs)
3. Plausible alternative explanation
4. Immediate reversible containment
5. Evidence to collect next
6. Human decisions required

Do not execute or claim to execute any action."""

response = client.responses.create(
    model=MODEL_ID,
    instructions=instructions,
    input=task,
    max_output_tokens=MAX_OUTPUT_TOKENS,
    store=False,
)

output_text = response.output_text.strip()
if response.status != "completed" or not output_text:
    raise RuntimeError(
        f"Incident-triage request did not complete: id={response.id}, "
        f"status={response.status!r}, details={response.incomplete_details!r}"
    )

print(output_text)

## Review and conclude

### What was achieved

- Discovery found the exact model ID, and the access check confirmed that the selected identity can invoke Daybreak Blue through the Ohio Bedrock Mantle endpoint.
- Five synthetic events were converted into an assessment, evidence-linked timeline, alternative explanation, reversible containment options, missing-evidence list, and explicit human decisions.
- The workflow remained advisory: it used no tools and took no containment action.

### Decide whether the answer is usable

Check that every material claim points back to an event ID, facts and inferences are kept separate, confidence is justified, alternatives are plausible, and proposed containment is reversible. The missing-evidence section should tell an analyst what to collect next rather than filling gaps with assumptions.

### What this proof of concept does not establish

The model response is an investigation hypothesis, not an incident declaration. It has not checked live telemetry, confirmed identity ownership, or executed a response action. The useful outcome is a traceable brief that helps a human analyst decide what to verify and do next.